# Train the action-conditioned win model on an A100

This notebook mounts Google Drive, uses the existing nine-feature cache, clones the latest project code, and trains the battle-11 win model from random initialization. Checkpoints and results are written to Drive so interrupted runs can resume.

In Colab, select **Runtime > Change runtime type > A100 GPU**, then run all cells.

In [ ]:
from pathlib import Path

DRIVE_PROJECT_FOLDER = Path("clash2")  # Relative to MyDrive.
REPOSITORY_URL = "https://github.com/jfbami/clash2.git"
BRANCH = "main"
MAX_EPOCHS = 60
PATIENCE = 8
MIN_DELTA = 1e-4
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
SEEDS = [17]  # Add seeds only if a separate multi-seed run is approved.
REQUIRE_A100 = True

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive") / DRIVE_PROJECT_FOLDER
DRIVE_CACHE = DRIVE_PROJECT_ROOT / "data/switch_current_deck_count_ablation/arrays"
OUTPUT_DIR = DRIVE_PROJECT_ROOT / "data/outcome_baseline"
required_cache_files = [
    DRIVE_CACHE / "metadata.json",
    DRIVE_CACHE / "cards.npy",
    DRIVE_CACHE / "levels.npy",
    DRIVE_CACHE / "battle_features.npy",
    DRIVE_CACHE / "summary_features.npy",
    DRIVE_CACHE / "labels.npy",
    DRIVE_CACHE / "next_wins.npy",
    DRIVE_CACHE / "splits.npy",
    DRIVE_CACHE / "player_indices.npy",
]
missing = [str(path) for path in required_cache_files if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required Drive cache files:\n" + "\n".join(missing))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Drive cache: {DRIVE_CACHE}")
print(f"Persistent output: {OUTPUT_DIR}")

In [ ]:
import shutil
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select an A100 GPU runtime before continuing.")
gpu_name = torch.cuda.get_device_name(0)
if REQUIRE_A100 and "A100" not in gpu_name.upper():
    raise RuntimeError(f"Expected an A100 runtime, but Colab provided: {gpu_name}")
print(f"GPU: {gpu_name}")

CODE_ROOT = Path("/content/clash2_code")
if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(CODE_ROOT)],
    check=True,
)
commit = subprocess.run(
    ["git", "-C", str(CODE_ROOT), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
required_code = [
    CODE_ROOT / "models/outcome_model.py",
    CODE_ROOT / "scripts/train_outcome_baseline.py",
]
missing_code = [str(path) for path in required_code if not path.exists()]
if missing_code:
    raise FileNotFoundError("Cloned repository is missing required code:\n" + "\n".join(missing_code))
print(f"Code ready at commit {commit}: {CODE_ROOT}")

In [ ]:
import json

metadata = json.loads((DRIVE_CACHE / "metadata.json").read_text(encoding="utf-8"))
if metadata.get("continuity") != "same_collection":
    raise ValueError("The cache must use continuity='same_collection'.")
if metadata.get("summary_feature_set") != "current_deck_count":
    raise ValueError("The cache must contain the selected nine long-term features.")

SESSION_CACHE = Path("/content/clash2_current_deck_count_arrays")
source_metadata = (DRIVE_CACHE / "metadata.json").read_bytes()
cache_is_current = (
    (SESSION_CACHE / "metadata.json").exists()
    and (SESSION_CACHE / "metadata.json").read_bytes() == source_metadata
)
if not cache_is_current:
    if SESSION_CACHE.exists():
        shutil.rmtree(SESSION_CACHE)
    shutil.copytree(DRIVE_CACHE, SESSION_CACHE)
print(f"Session cache ready: {SESSION_CACHE}")
print(json.dumps({
    "examples": metadata.get("examples"),
    "players": metadata.get("players"),
    "summary_feature_set": metadata.get("summary_feature_set"),
    "splits": metadata.get("splits"),
}, indent=2))

## Train

This runs one approved random initialization by default. The training script resumes from Drive checkpoints if the runtime disconnects.

In [ ]:
import sys

command = [
    sys.executable, "-u", str(CODE_ROOT / "scripts/train_outcome_baseline.py"),
    "--cache", str(SESSION_CACHE),
    "--output-dir", str(OUTPUT_DIR),
    "--device", "cuda",
    "--max-epochs", str(MAX_EPOCHS),
    "--patience", str(PATIENCE),
    "--min-delta", str(MIN_DELTA),
    "--batch-size", str(BATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE),
    "--weight-decay", str(WEIGHT_DECAY),
    "--seeds", *map(str, SEEDS),
]
print("Running:", " ".join(command))
subprocess.run(command, check=True)

## Inspect factual validation and test results

These metrics evaluate the win prediction only under the action each player actually took. They do not validate the unobserved counterfactual outcome.

In [ ]:
import pandas as pd
from IPython.display import display

report = json.loads((OUTPUT_DIR / "results.json").read_text(encoding="utf-8"))
rows = []
for run in report["runs"]:
    for split in ("validation", "test"):
        for group in ("overall", "stay", "switch"):
            result = run[split][group]
            rows.append({
                "seed": run["seed"],
                "best_epoch": run["best_epoch"],
                "split": split,
                "observed_action": group,
                **result,
            })
display(pd.DataFrame(rows).set_index(["seed", "split", "observed_action"]).round(6))
print(f"Results: {OUTPUT_DIR / 'results.json'}")

All checkpoints, progress state, and results remain under `MyDrive/clash2/data/outcome_baseline`. Re-running the notebook resumes an interrupted run when the settings and cache metadata match.